# Market Data (Candles) Ingestion Test

This notebook tests and validates the historical candle ingestion pipeline:
- Checks current data coverage
- Identifies gaps and missing candles
- Generates commands to fill missing data
- Validates data integrity

**Date**: 2026-08-26  
**Database**: fin-market-db (Azure SQL)  
**Schema**: crypto.market_prices

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone
import pandas as pd
from sqlalchemy import create_engine, text, func
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

from cryptoquant.database.session import get_session
from cryptoquant.database.models import MarketPrice, TradingPair, TrackedPair
from cryptoquant.config.settings import get_settings

print("✓ Imports successful")
print(f"Current time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")

## 2. Database Connection

In [ ]:
session = get_session()
settings = get_settings()

print(f"✓ Connected to: {settings.db_server}")
print(f"✓ Database: {settings.db_name}")
print(f"✓ Schema: crypto")

## 3. Current Data Overview

In [ ]:
query = text("""
SELECT 
    COUNT(*) AS total_candles,
    MIN(timestamp) AS first_candle,
    MAX(timestamp) AS latest_candle,
    DATEDIFF(DAY, MIN(timestamp), MAX(timestamp)) AS days_span,
    COUNT(DISTINCT trading_pair_id) AS distinct_pairs
FROM crypto.market_prices
""")

result = session.execute(query).fetchone()
df_overview = pd.DataFrame([result._mapping], columns=result._mapping.keys())

print("=== Market Prices Overview ===")
display(df_overview)

total = result.total_candles
print(f"\n📊 Total candles: {total:,}")

## 4. Coverage by Trading Pair

In [ ]:
query = text("""
SELECT
    tp.symbol AS currency_pair,
    COUNT(*) AS actual_candles,
    MIN(mp.timestamp) AS first_candle,
    MAX(mp.timestamp) AS last_candle,
    DATEDIFF(HOUR, MIN(mp.timestamp), MAX(mp.timestamp)) + 1 AS expected_candles,
    DATEDIFF(HOUR, MIN(mp.timestamp), MAX(mp.timestamp)) + 1 - COUNT(*) AS missing_candles,
    CAST(COUNT(*) * 100.0 / NULLIF(DATEDIFF(HOUR, MIN(mp.timestamp), MAX(mp.timestamp)) + 1, 0) AS DECIMAL(5,2)) AS coverage_percent
FROM crypto.market_prices AS mp
INNER JOIN crypto.trading_pairs AS tp ON tp.id = mp.trading_pair_id
GROUP BY tp.symbol
ORDER BY coverage_percent ASC
""")

df_coverage = pd.DataFrame(session.execute(query).fetchall(), 
                           columns=['currency_pair', 'actual_candles', 'first_candle', 
                                   'last_candle', 'expected_candles', 'missing_candles', 'coverage_percent'])

print("=== Coverage by Trading Pair ===")
display(df_coverage)

# Highlight issues
issues = df_coverage[df_coverage['coverage_percent'] < 99.0]
if len(issues) > 0:
    print(f"\n⚠️  {len(issues)} pair(s) with < 99% coverage - needs attention")
else:
    print("\n✓ All pairs have >= 99% coverage")

## 5. Identify Gaps (> 1 hour)

In [ ]:
query = text("""
WITH ordered_candles AS (
    SELECT
        mp.trading_pair_id,
        tp.symbol,
        mp.timestamp,
        LEAD(mp.timestamp) OVER (PARTITION BY mp.trading_pair_id ORDER BY mp.timestamp) AS next_timestamp
    FROM crypto.market_prices AS mp
    INNER JOIN crypto.trading_pairs AS tp ON tp.id = mp.trading_pair_id
),
gaps AS (
    SELECT
        symbol,
        DATEDIFF(HOUR, timestamp, next_timestamp) AS hours_gap
    FROM ordered_candles
    WHERE next_timestamp IS NOT NULL
      AND DATEDIFF(HOUR, timestamp, next_timestamp) > 1
)
SELECT
    symbol AS currency_pair,
    COUNT(*) AS gap_count,
    SUM(hours_gap) AS total_hours_missing,
    MIN(hours_gap) AS min_gap_hours,
    MAX(hours_gap) AS max_gap_hours,
    AVG(hours_gap) AS avg_gap_hours
FROM gaps
GROUP BY symbol
ORDER BY total_hours_missing DESC
""")

df_gaps = pd.DataFrame(session.execute(query).fetchall(),
                       columns=['currency_pair', 'gap_count', 'total_hours_missing',
                               'min_gap_hours', 'max_gap_hours', 'avg_gap_hours'])

if len(df_gaps) > 0:
    print("=== Data Gaps Summary ===")
    display(df_gaps)
    total_missing = df_gaps['total_hours_missing'].sum()
    print(f"\n⚠️  Total missing hours across all pairs: {total_missing:,.0f}")
else:
    print("✓ No gaps detected - all hourly records present!")

## 6. Generate Re-Ingestion Commands

Based on the coverage analysis, generate PowerShell commands to fill missing data.

In [ ]:
print("=== Re-Ingestion Commands ===\n")
print("Copy and paste these commands in PowerShell:\n")
print("cd D:\\data\\development\\crypto")
print(".\.venv\Scripts\Activate.ps1\n")

for idx, row in df_coverage.iterrows():
    pair = row['currency_pair']
    coverage = row['coverage_percent']
    missing = row['missing_candles']
    
    if coverage < 90:
        # High priority - full historical ingestion (2 years)
        days = 730
        priority = "HIGH PRIORITY"
    elif coverage < 95:
        # Medium priority - 1 year
        days = 365
        priority = "MEDIUM"
    elif coverage < 99:
        # Low priority - 90 days
        days = 90
        priority = "LOW"
    else:
        continue  # Good coverage, skip
    
    print(f"# {priority}: {pair} ({coverage}% coverage, {missing} missing)")
    print(f"python scripts/collect_historic_data.py --granularity hourly --days {days} --product-id {pair}")
    print()

if df_coverage['coverage_percent'].min() >= 99:
    print("✓ All pairs have >= 99% coverage - No re-ingestion needed!")

## 7. Historical Ingestion Command (All Pairs)

If you want to re-ingest ALL pairs for a specific time period:

In [ ]:
print("=== Full Historical Ingestion Commands ===\n")
print("# 30 days (all pairs)")
print("python scripts/collect_historic_data.py --granularity hourly --days 30\n")

print("# 90 days (all pairs)")
print("python scripts/collect_historic_data.py --granularity hourly --days 90\n")

print("# 1 year (all pairs)")
print("python scripts/collect_historic_data.py --granularity hourly --days 365\n")

print("# 2 years (all pairs)")
print("python scripts/collect_historic_data.py --granularity hourly --days 730\n")

print("# 3 years (all pairs)")
print("python scripts/collect_historic_data.py --granularity hourly --days 1095\n")

## 8. Cleanup + Full Re-Ingestion Commands

If you need to clean up and start fresh:

In [ ]:
print("=== Cleanup + Re-Ingestion Pipeline ===\n")
print("⚠️  WARNING: This will delete ALL candle and technical analysis data!\n")
print("Steps:")
print("1. Run cleanup SQL script (tests/sql/cleanup_candles.sql)")
print("   - This deletes technical_analysis first, then market_prices")
print("   - Remember to change ROLLBACK to COMMIT to save changes\n")

print("2. Run historical candle ingestion")
print("python scripts/collect_historic_data.py --granularity hourly --days 730\n")

print("3. Run technical analysis after candles complete")
print("python scripts/calculate_technical_analysis.py --mode historical --days 730\n")

print("Combined PowerShell script:")
print("-" * 60)
print("""# Step 1: Run cleanup SQL manually in Azure Data Studio
# Edit tests/sql/cleanup_candles.sql and change ROLLBACK to COMMIT

# Step 2: Activate environment
cd D:\\data\\development\\crypto
.\\.venv\\Scripts\\Activate.ps1

# Step 3: Ingest candles (2 years)
Write-Host "`n=== Ingesting Candles ===" -ForegroundColor Cyan
python scripts/collect_historic_data.py --granularity hourly --days 730

# Step 4: Calculate technical analysis
Write-Host "`n=== Calculating Technical Analysis ===" -ForegroundColor Cyan
python scripts/calculate_technical_analysis.py --mode historical --days 730

Write-Host "`n✓ Pipeline complete!" -ForegroundColor Green
""")

## 9. Validation Summary

In [ ]:
print("=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

total_candles = df_overview['total_candles'][0]
min_coverage = df_coverage['coverage_percent'].min()
avg_coverage = df_coverage['coverage_percent'].mean()
pairs_with_issues = len(df_coverage[df_coverage['coverage_percent'] < 99.0])

print(f"\n📊 Total Candles: {total_candles:,}")
print(f"📈 Coverage Range: {min_coverage:.2f}% - 100%")
print(f"📈 Average Coverage: {avg_coverage:.2f}%")
print(f"⚠️  Pairs Needing Attention: {pairs_with_issues}")

if len(df_gaps) > 0:
    print(f"⚠️  Total Gaps: {len(df_gaps)} pair(s) have gaps > 1 hour")
else:
    print("✓ No gaps detected")

print("\n" + "=" * 70)

if min_coverage >= 99.0 and len(df_gaps) == 0:
    print("✅ STATUS: EXCELLENT - Data is clean and complete")
elif min_coverage >= 95.0:
    print("⚠️  STATUS: GOOD - Minor gaps, low priority fixes needed")
elif min_coverage >= 90.0:
    print("⚠️  STATUS: NEEDS ATTENTION - Medium priority fixes needed")
else:
    print("❌ STATUS: CRITICAL - Major data gaps, high priority ingestion needed")

print("=" * 70)

## 10. Close Connection

In [ ]:
session.close()
print("✓ Database connection closed")